# Sentence Embeddings

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "How can I buy a car?",
    "What are the steps for purchasing a vehicle?",
    "I like eating pizza"
]

embeddings = model.encode(sentences)

print("Shape:", embeddings.shape)
print("\nFirst sentence embedding:")
print(embeddings[0])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Dell\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape: (3, 384)

First sentence embedding:
[-1.49094192e-02  3.32658403e-02  1.67909954e-02 -5.78299584e-03
 -7.11592212e-02  5.08439243e-02  1.57571286e-02  7.66343772e-02
 -1.13901254e-02  2.29245275e-02  2.08505969e-02 -2.52142828e-02
 -4.78584814e-04  2.36025192e-02  1.83195136e-02 -6.83739185e-02
 -3.70465815e-02  7.67577998e-03  2.49311477e-02 -1.56056909e-02
 -4.65766750e-02 -1.85405184e-02 -2.15184037e-02 -4.06166539e-02
 -2.43159235e-02 -1.94171928e-02 -8.08629245e-02  2.13535242e-02
  1.15281101e-02  4.50427532e-02  7.68776238e-02  1.72467460e-03
 -3.08901910e-02  3.68032255e-03  4.85204197e-02 -7.45812282e-02
  1.13312183e-02 -5.92172891e-02  7.43231829e-03  5.29484637e-03
  3.42675224e-02 -6.35313094e-02 -7.35351965e-02  2.73158830e-02
  3.34612392e-02  1.32042530e-03  7.80843571e-02  1.42196566e-01
  1.23172440e-01 -1.78527907e-02 -2.44490448e-02  6.81787133e-02
 -6.92941062e-03 -7.05569386e-02 -1.79073170e-01 -1.59264468e-02
 -6.50873482e-02  7.05793202e-02  1.65171232e-0

 - Sentence
   ↓
- Embedding Model
   ↓
 - Vector

# Cosine Similarity

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    [embeddings[0]],
    [embeddings[1]]
)

print("Similarity:", similarity[0][0])

Similarity: 0.7182054


In [4]:
similarity2 = cosine_similarity(
    [embeddings[0]],
    [embeddings[2]]
)

print("Similarity:", similarity2[0][0])

Similarity: 0.09311354


# Semantic Search

In [5]:
documents = [
    "How to apply for a bank loan",
    "How to cook pasta",
    "Steps to purchase a car",
    "How to train a machine learning model"
]

doc_embeddings = model.encode(documents)

query = "I want to buy a vehicle"

query_embedding = model.encode([query])

scores = cosine_similarity(query_embedding, doc_embeddings)[0]

for doc, score in zip(documents, scores):
    print(f"{score:.4f} -> {doc}")

0.2361 -> How to apply for a bank loan
0.0170 -> How to cook pasta
0.6838 -> Steps to purchase a car
0.1125 -> How to train a machine learning model


In [6]:
best_index = scores.argmax()

print("Query:", query)
print("Best document:", documents[best_index])
print("Score:", scores[best_index])

Query: I want to buy a vehicle
Best document: Steps to purchase a car
Score: 0.6837626


# Vector Database / FAISS

In [7]:
import faiss
import numpy as np

document_embeddings = model.encode(documents)

document_embeddings = np.array(document_embeddings).astype("float32")

dimension = document_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(document_embeddings)

print("Number of vectors:", index.ntotal)

Number of vectors: 4


In [8]:
query = "I want to purchase a vehicle"

query_embedding = model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

distances, indices = index.search(query_embedding, k=2)

print("Top results:")

for idx in indices[0]:
    print(documents[idx])

Top results:
Steps to purchase a car
How to apply for a bank loan
